# Fiddler Experiment Trace Capture Quick Start

## Goal

A score tells you **that** an item failed. The trace tells you **why**.

When you run an experiment, the Fiddler Evals SDK captures the OpenTelemetry spans your task emits
and links them to the experiment item that produced them. You get per-item traces in the UI for
debugging, and your evaluators can score on the **execution itself** — tool call order, retry
counts, token usage, latency — not just the final output string.

### What you'll do

1. Connect to Fiddler and set up a pre-production application
2. Write an instrumented task that emits spans
3. Run an experiment and watch traces get captured automatically
4. Write evaluators that score on the captured trace
5. Understand the app guard that keeps eval traces out of production

### Prerequisites

- A Fiddler API key (**Settings > Credentials**)
- Python 3.10+

There is **nothing to turn on**. `fiddler-otel` is a base dependency of the Evals SDK, so
`evaluate()` sets capture up on its own as long as it can resolve a Fiddler application for the
dataset.

## 0. Imports and Configuration

In [ ]:
# Install the Fiddler Evaluations SDK (fiddler-otel comes with it)
%pip install -q --upgrade fiddler-evals

import time

from fiddler_evals import (
    Application,
    Dataset,
    EvalFn,
    NewDatasetItem,
    Project,
    Score,
    __version__,
    evaluate,
    init,
)
from fiddler_evals.evaluators import AnswerRelevance

# fiddler-otel gives us the tracing primitives used to instrument the task
from fiddler_otel import get_client, get_current_span, trace

print(f'Fiddler Evals SDK version: {__version__}')

## 1. Connect to Fiddler

**What you need:**

1. **Fiddler URL** — your instance URL (e.g. `https://your-org.fiddler.ai`)
2. **API Key** — found in the **Credentials** tab on your Fiddler **Settings** page

In [ ]:
# Replace with your Fiddler instance details
URL = ''  # Full URL including https:// (e.g. 'https://your_company_name.fiddler.ai')
API_KEY = ''  # Your Fiddler API key from Settings > Credentials

# LLM Gateway configuration (Settings > LLM Gateway).
# OPTIONAL here -- the trace-based evaluators in this notebook need no LLM.
# Set these only if you also want an LLM-as-a-judge evaluator alongside them.
LLM_MODEL_NAME = ''  # e.g. 'openai/gpt-4o-mini'
LLM_CREDENTIAL_NAME = ''  # The credential name from Settings > LLM Gateway

**Use a dedicated pre-production application.**

Eval traces are real traces. Sending them to your production application would pollute production
monitoring with synthetic traffic, so keep evaluation in its own application. Section 6 covers the
guard that enforces this.

In [ ]:
PROJECT_NAME = 'eval_demos'
APPLICATION_NAME = 'support-agent-preprod'  # pre-production, NOT your prod app
DATASET_NAME = 'trace_capture_demo'

In [ ]:
init(url=URL, token=API_KEY)

project = Project.get_or_create(name=PROJECT_NAME)
application = Application.get_or_create(name=APPLICATION_NAME, project_id=project.id)

print(f'Project:     {project.name} ({project.id})')
print(f'Application: {application.name} ({application.id})')

## 2. Build a small dataset

In [ ]:
dataset = Dataset.get_or_create(
    name=DATASET_NAME,
    application_id=application.id,
    description='Small dataset for demonstrating eval-run trace capture',
)

items = [
    NewDatasetItem(
        inputs={'user_query': 'How do I reset my password?'},
        expected_outputs={'expected_response': 'Use the Forgot Password link on the sign-in page.'},
    ),
    NewDatasetItem(
        inputs={'user_query': 'What is your refund policy?'},
        expected_outputs={'expected_response': 'Refunds are available within 30 days of purchase.'},
    ),
    NewDatasetItem(
        inputs={'user_query': 'Do you support SSO?'},
        expected_outputs={'expected_response': 'Yes, SAML and OIDC are both supported.'},
    ),
]

if not list(dataset.get_items()):
    dataset.insert(items=items)

print(f'Dataset: {dataset.name} ({dataset.id}) — {len(list(dataset.get_items()))} items')

## 3. Write an instrumented task

This stands in for your real agent. The important part is that it **emits spans** — here via
`fiddler-otel`'s `@trace` decorator, which creates a span per call and records inputs and outputs.

`@trace` resolves the Fiddler client at **call time**, not at decoration time, so it works inside
`evaluate()`: the Evals SDK has already created the client by the time your task runs. You do not
need to wire anything up to the experiment — the SDK stamps every span your task produces with the
experiment and item IDs automatically.

Note the task signature must be exactly `(inputs, extras, metadata)`. `@trace` uses
`functools.wraps`, so the decorated task still presents that signature to the runner's validation.

In [ ]:
KNOWLEDGE_BASE = {
    'password': 'Use the Forgot Password link on the sign-in page.',
    'refund': 'Refunds are available within 30 days of purchase.',
    'sso': 'Yes, SAML and OIDC are both supported.',
}


@trace(as_type='tool')
def search_kb(query):
    """A 'tool' the agent calls. Emits a span with fiddler.span.type == 'tool'."""
    span = get_current_span(as_type='tool')

    for keyword, answer in KNOWLEDGE_BASE.items():
        if keyword in query.lower():
            if span:
                span.set_attribute('kb.hit', True)
            return answer

    if span:
        span.set_attribute('kb.hit', False)
    return None


@trace(as_type='chain')
def support_agent(inputs, extras, metadata):
    """Eval task. Signature must be exactly (inputs, extras, metadata)."""
    user_query = inputs['user_query']

    answer = search_kb(user_query)

    # Simulate a retry when the first lookup misses.
    if answer is None:
        answer = search_kb(user_query.split()[-1])

    if answer is None:
        answer = "I don't have an answer for that yet."

    time.sleep(0.05)  # stand in for model latency
    return {'rag_response': answer}

## 4. Run the experiment

Trace capture is automatic. Run the experiment exactly as you normally would.

In [ ]:
# No evaluators yet -- this run is just to show that traces are captured
# and linked with zero setup.
result = evaluate(
    dataset=dataset,
    task=support_agent,
    evaluators=[],
    name_prefix='trace_capture_demo',
)

experiment = result.experiment
print(f'Experiment: {experiment.name} ({experiment.id})')
print(f'Status:     {experiment.status}')

### How linking works

The SDK mints one `experiment_item_id` per dataset item and uses it as the session identifier —
the two are 1:1. For each item, before your task runs, the SDK:

1. Sets `gen_ai.conversation.id` to the `experiment_item_id` on every span the item produces, so the
   item's traces are grouped under an ID Fiddler owns.
2. Stamps two attributes on every span, cascading from parent to child:

| Span attribute | Value |
|---|---|
| `fiddler.session.user.experiment_id` | The experiment's ID |
| `fiddler.session.user.experiment_item_id` | The item's ID |

3. Opens an in-memory buffer keyed by `experiment_item_id`.

When the task returns — **or raises** — the buffer is drained and handed to your evaluators. A
failing task still yields whatever spans it produced before the exception, which is usually the
interesting case.

Because spans are bucketed by `fiddler.session.user.experiment_item_id` rather than by
`gen_ai.conversation.id`, capture stays correct even if your agent sets its own conversation ID.

No separate session ID is persisted on the experiment item row — the UI resolves an item's traces by
`experiment_item_id`, because the session ID **is** the item ID.

## 5. Score an evaluator on the trace

Declare a `session` parameter on your evaluator's score function and the runner passes the captured
spans to it. Parameter binding is **by name**, so evaluators that do not declare `session` are
unaffected.

Always default `session` to `None` and handle the `None` case. Capture is best-effort, so an
evaluator that assumes a session breaks the moment tracing is unavailable.

### What a session contains

Your evaluator receives a `Session` with two attributes:

- **`session_id`** (`UUID`) — the experiment item's ID
- **`spans`** (`list[dict]`) — the spans the task produced, in completion order

Each span is a plain dict — no OpenTelemetry objects to import:

| Key | Type | Notes |
|---|---|---|
| `trace_id` | `str` | Hex, `0x`-prefixed |
| `span_id` | `str` | Hex, `0x`-prefixed |
| `parent_span_id` | `str \| None` | `None` for root spans |
| `name` | `str` | Span name |
| `kind` | `str \| None` | `INTERNAL`, `CLIENT`, `SERVER`, … |
| `start_time` | `int` | **Nanoseconds** |
| `end_time` | `int` | **Nanoseconds** |
| `status_code` | `str \| None` | `OK`, `ERROR`, `UNSET` |
| `status_description` | `str \| None` | |
| `attributes` | `dict` | Where instrumentations put payloads |
| `events` | `list[dict]` | Each has `name`, `timestamp`, `attributes` |

Times are integer nanoseconds so duration is a plain subtraction with no precision loss.

In [ ]:
def inspect_trace(session=None):
    """Print the captured span tree. Returns None so the score is SKIPPED."""
    if session is None:
        print('No trace captured')
        return None

    print(f'session_id={session.session_id}  spans={len(session.spans)}')
    for span in session.spans:
        duration_ms = (span['end_time'] - span['start_time']) / 1_000_000
        parent = span['parent_span_id'] or '-'
        print(f"  {span['name']:<20} {duration_ms:>7.1f}ms  status={span['status_code']}  parent={parent}")
    return None


result = evaluate(
    dataset=dataset,
    task=support_agent,
    evaluators=[EvalFn(inspect_trace, score_name='inspect_trace')],
    name_prefix='trace_capture_inspect',
)
print(f'\nExperiment: {result.experiment.name} ({result.experiment.id})')

### Trace-based evaluators

`EvalFn` converts your return value for you:

- `bool` becomes 1.0 or 0.0
- `int` and `float` pass through
- `None` produces a **SKIPPED** score
- a `Score` you build yourself is used as-is

Three evaluators you cannot write without the trace:

In [ ]:
def tool_call_efficiency(session=None):
    """Score 1.0 when the agent used three or fewer tool calls."""
    if session is None:
        return None

    tool_spans = [
        span for span in session.spans
        if span['attributes'].get('fiddler.span.type') == 'tool'
    ]
    return len(tool_spans) <= 3


def responded_within_3s(session=None):
    """Score on end-to-end latency measured from the root span."""
    if session is None or not session.spans:
        return None

    root = next(
        (span for span in session.spans if span['parent_span_id'] is None),
        session.spans[-1],
    )
    duration_ms = (root['end_time'] - root['start_time']) / 1_000_000
    return duration_ms < 3000


def no_swallowed_exceptions(session=None):
    """Exceptions arrive as span EVENTS, not attributes."""
    if session is None:
        return None

    exceptions = [
        event
        for span in session.spans
        for event in span['events']
        if event['name'] == 'exception'
    ]
    return not exceptions

Return a `Score` directly when you want to control the reasoning text. A `Score` you construct
yourself requires both `name` and `evaluator_name`, and the explanation field is `reasoning`.

In [ ]:
def kb_hit_rate(session=None):
    """Did the knowledge-base lookup actually hit?"""
    if session is None:
        return None

    hits = [
        span for span in session.spans
        if span['attributes'].get('kb.hit') is True
    ]
    misses = [
        span for span in session.spans
        if span['attributes'].get('kb.hit') is False
    ]

    return Score(
        name='kb_hit_rate',
        evaluator_name='kb_hit_rate',
        value=1.0 if hits else 0.0,
        reasoning=f'{len(hits)} hits, {len(misses)} misses',
    )

In [ ]:
# These four score purely on the captured trace -- no LLM required.
evaluators = [
    EvalFn(tool_call_efficiency, score_name='tool_call_efficiency'),
    EvalFn(responded_within_3s, score_name='responded_within_3s'),
    EvalFn(no_swallowed_exceptions, score_name='no_swallowed_exceptions'),
    EvalFn(kb_hit_rate, score_name='kb_hit_rate'),
]

# LLM-as-a-judge evaluators additionally need an LLM Gateway model.
if LLM_MODEL_NAME:
    evaluators.append(
        AnswerRelevance(model=LLM_MODEL_NAME, credential=LLM_CREDENTIAL_NAME)
    )

result = evaluate(
    dataset=dataset,
    task=support_agent,
    evaluators=evaluators,
    name_prefix='trace_based_evaluators',
)

experiment = result.experiment
print(f'Experiment: {experiment.name} ({experiment.id})')
print(f'Status:     {experiment.status}')
print(f'Scored {len(result.results)} items')
print('\nOpen the experiment in Fiddler to see per-item traces alongside the scores.')

<div class="alert alert-warning">

**`session` is a reserved parameter name.** The runner binds it *after* spreading your task's
outputs, so a task output named `session` cannot shadow it — and cannot be read by an evaluator that
expects it. Rename that output if you have one.

</div>

## 6. Keeping eval traces out of production

A single process shares one global `FiddlerClient`, and eval traces are real traces.

The SDK guards against mixing them with production. If a `FiddlerClient` already exists and its
`application_id` does not match the dataset's application, `evaluate()` raises `ValueError` rather
than degrading quietly:

```
An existing FiddlerClient is scoped to application '<prod-app-id>', but the
dataset's application is '<eval-app-id>'. Point the FiddlerClient at the
dataset's (pre-production) application to avoid sending eval traces to the
wrong application.
```

This is the **one** capture failure that is not best-effort, because silently mixing evaluation and
production traffic is worse than a failed run. Everything else — a missing client, a tracing backend
that is down — logs a warning and lets the experiment run normally.

Because the client is a process-global singleton, one process can run experiments for **one**
application at a time. To evaluate several applications, use separate processes.

In [ ]:
# Confirm which application eval traces are going to.
try:
    client = get_client()
    print(f'FiddlerClient application_id: {client.application_id}')
    print(f'Dataset application_id:       {application.id}')
    match = str(client.application_id) == str(application.id)
    print(f'\nMatch: {match}' if match else '\nMISMATCH — evaluate() would raise ValueError')
except RuntimeError:
    print('No FiddlerClient yet — the SDK will build one scoped to the dataset application.')

### Concurrency notes

Capture is safe under the runner's multi-threaded execution. `set_session()` runs inside each worker
thread and spans are bucketed per experiment item, so concurrent items never cross-contaminate.

One capture processor is registered per `FiddlerClient` and reused across runs, rather than one per
`evaluate()` call. OpenTelemetry has no `remove_span_processor`, so reuse is what keeps a
long-running process from accumulating processors.

The processor only buffers spans for items the runner has explicitly opened. Late or orphaned spans
are dropped immediately rather than creating a bucket that is never freed, which keeps memory
bounded.

At the end of a run the SDK flushes the client's span exporters so traces ship promptly instead of
waiting on the batch timer. It never shuts the client down, since the client may be shared.

## Congratulations!

You've used eval-run trace capture end to end:

- **Ran** an experiment with an instrumented task — no tracing setup required
- **Inspected** the captured span tree per experiment item
- **Scored** evaluators on tool calls, latency, and span events
- **Verified** the app guard that keeps eval traces out of production

### Next steps

- [Capture traces during experiments](https://docs.fiddler.ai/evaluate-and-test/eval-trace-capture)
- [Build a golden dataset from production spans](https://docs.fiddler.ai/evaluate-and-test/golden-datasets)
- [Evals SDK Quick Start](https://docs.fiddler.ai/evaluate-and-test/evals-sdk-quick-start)

### Questions?

Reach out to us at [help@fiddler.ai](mailto:help@fiddler.ai).